# Alpha Decay Analysis
**AlphaMind Research | Quantitative Strategy Research**

---

## Overview

This notebook provides a comprehensive, production-grade analysis of alpha signal decay
for AlphaMind strategies. It quantifies how quickly each signal's predictive power
attenuates over multiple forecast horizons and derives the optimal rebalancing frequency.

### Sections
1. [Environment Setup](#1-environment-setup)
2. [Data Generation / Loading](#2-data-generation--loading)
3. [Signal Pre-processing & Neutralization](#3-signal-pre-processing--neutralization)
4. [Information Coefficient (IC) Analysis](#4-information-coefficient-ic-analysis)
5. [IC Information Ratio (ICIR)](#5-ic-information-ratio-icir)
6. [Alpha Decay Curve & Half-Life Estimation](#6-alpha-decay-curve--half-life-estimation)
7. [Signal Autocorrelation & Persistence](#7-signal-autocorrelation--persistence)
8. [Turnover & Cost-Adjusted IC](#8-turnover--cost-adjusted-ic)
9. [Rolling IC & Regime Overlay](#9-rolling-ic--regime-overlay)
10. [Optimal Holding Period](#10-optimal-holding-period)
11. [Multi-Signal Ensemble Decay](#11-multi-signal-ensemble-decay)
12. [Summary Dashboard](#12-summary-dashboard)

> **Data note**: Synthetic data is used here to mirror realistic cross-sectional equity  
> signal distributions. Replace the `DataLoader` section with live backtest outputs.


## 1. Environment Setup

In [ ]:
from __future__ import annotations

# - Standard library
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# - Numerical / data
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit
from scipy.stats import spearmanr, pearsonr, t as t_dist

# - Visualisation─
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# - Styling─
plt.style.use("seaborn-v0_8-darkgrid")
COLORS = {
    "blue":    "#2563eb",
    "red":     "#ef4444",
    "green":   "#10b981",
    "orange":  "#f59e0b",
    "purple":  "#8b5cf6",
    "cyan":    "#06b6d4",
    "pink":    "#ec4899",
    "gray":    "#6b7280",
}
sns.set_palette(list(COLORS.values()))

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)

print("✅  Libraries loaded - AlphaMind Alpha Decay Research v2.0")


## 2. Data Generation / Loading

We simulate a realistic **cross-sectional equity universe** (500 assets x 4 years of daily data).  
Each signal type has a distinct decay profile, mimicking real-world categories:

| Signal | Category | Decay Speed |
|--------|----------|-------------|
| `momentum_1m` | Price momentum | Medium |
| `value_bp`    | Book-to-price  | Slow   |
| `quality_roe` | Return-on-equity | Slow  |
| `reversal_5d` | Short-term reversal | Fast |
| `sentiment`   | News sentiment    | Fast  |


In [ ]:
@dataclass
class AlphaDecayConfig:
    """Central configuration for the alpha decay study."""
    n_assets:   int   = 500
    n_days:     int   = 1008          # ~4 years of trading days
    start_date: str   = "2020-01-02"
    horizons:   List[int] = field(default_factory=lambda: [1, 3, 5, 10, 21, 42, 63])
    rolling_window: int   = 63        # 3-month rolling IC window
    seed:       int   = 42
    bps_cost:   float = 5.0           # one-way transaction cost in bps

CFG = AlphaDecayConfig()
rng = np.random.default_rng(CFG.seed)

# - Calendar
dates  = pd.bdate_range(CFG.start_date, periods=CFG.n_days)
assets = [f"ASSET_{i:04d}" for i in range(CFG.n_assets)]

# - Signal definitions (name -> decay_lambda, IC_0, autocorrelation_rho)
SIGNAL_SPECS = {
    "momentum_1m": dict(ic0=0.060, lam=0.030, rho=0.92),
    "value_bp":    dict(ic0=0.045, lam=0.012, rho=0.98),
    "quality_roe": dict(ic0=0.040, lam=0.008, rho=0.99),
    "reversal_5d": dict(ic0=0.055, lam=0.180, rho=0.55),
    "sentiment":   dict(ic0=0.070, lam=0.250, rho=0.30),
}

def simulate_panel(spec: dict, cfg: AlphaDecayConfig) -> pd.DataFrame:
    """Simulate a cross-sectional signal + forward-return panel."""
    ic0, lam, rho = spec["ic0"], spec["lam"], spec["rho"]

    # AR(1) signal with cross-sectional noise
    raw = np.zeros((cfg.n_days, cfg.n_assets))
    raw[0] = rng.standard_normal(cfg.n_assets)
    for t in range(1, cfg.n_days):
        raw[t] = rho * raw[t - 1] + np.sqrt(1 - rho**2) * rng.standard_normal(cfg.n_assets)

    # Cross-sectional z-score (rank-normalised -> avoids outlier signal)
    def xs_zscore(arr):
        mu, sd = arr.mean(axis=1, keepdims=True), arr.std(axis=1, keepdims=True)
        return np.where(sd > 0, (arr - mu) / sd, 0.0)

    signal = xs_zscore(raw)

    panel = {f"signal": signal}

    common_noise = rng.standard_normal((cfg.n_days, cfg.n_assets))
    for h in cfg.horizons:
        strength = ic0 * np.exp(-lam * h)
        idio     = rng.standard_normal((cfg.n_days, cfg.n_assets))
        ret = strength * signal + np.sqrt(max(1 - strength**2, 0)) * (
            0.5 * common_noise + 0.5 * idio
        )
        panel[f"fwd_{h}d"] = ret

    idx = pd.MultiIndex.from_product([dates, assets], names=["date", "asset"])
    flat = {k: v.ravel() for k, v in panel.items()}
    df = pd.DataFrame(flat, index=idx).reset_index()
    return df

# - Build panels
panels: Dict[str, pd.DataFrame] = {}
for name, spec in SIGNAL_SPECS.items():
    panels[name] = simulate_panel(spec, CFG)
    print(f"  {name:15s}  shape={panels[name].shape}  "
          f"signal_std={panels[name]['signal'].std():.3f}")

print(f"\n✅  Panel generation complete - {CFG.n_assets} assets x {CFG.n_days} days x {len(SIGNAL_SPECS)} signals")


## 3. Signal Pre-processing & Neutralization

Industry standard pre-processing before IC computation:
- **Winsorize** extreme cross-sectional values (1st / 99th percentile)
- **Demean** within each date-sector group (sector-neutralization proxy)
- **Rank-transform** to uniform distribution (reduces sensitivity to outliers)


In [ ]:
def winsorize_xs(series: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    """Cross-sectional winsorisation within each date."""
    def _clip(g):
        lo, hi = g.quantile(lower), g.quantile(upper)
        return g.clip(lo, hi)
    return series.groupby(level="date").transform(_clip)


def rank_normalize_xs(series: pd.Series) -> pd.Series:
    """Cross-sectional rank -> uniform on [-0.5, +0.5] within each date."""
    def _rank(g):
        r = g.rank(method="average", na_option="keep")
        return (r - 1) / (len(r) - 1) - 0.5
    return series.groupby(level="date").transform(_rank)


def preprocess_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Apply full pre-processing pipeline to a signal panel."""
    out = df.set_index(["date", "asset"]).copy()
    out["signal_raw"] = out["signal"]
    out["signal"]     = winsorize_xs(out["signal"])
    out["signal"]     = rank_normalize_xs(out["signal"])
    return out.reset_index()


# Apply to all panels
processed: Dict[str, pd.DataFrame] = {}
for name, df in panels.items():
    processed[name] = preprocess_panel(df)

# Sanity check cross-sectional distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sample = processed["momentum_1m"]
sample_day = sample[sample["date"] == dates[100]]

axes[0].hist(sample_day["signal_raw"], bins=40, color=COLORS["blue"], alpha=0.7, edgecolor="white")
axes[0].set_title("Raw Signal Distribution (one date)", fontsize=12)
axes[0].set_xlabel("Signal value")

axes[1].hist(sample_day["signal"], bins=40, color=COLORS["green"], alpha=0.7, edgecolor="white")
axes[1].set_title("Rank-Normalised Signal Distribution (one date)", fontsize=12)
axes[1].set_xlabel("Rank-normalised value")

plt.suptitle("momentum_1m - Cross-Sectional Distribution Before vs After Normalisation",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("✅  Pre-processing complete")


## 4. Information Coefficient (IC) Analysis

**IC** measures the cross-sectional rank correlation between the signal and subsequent returns.

$$\text{IC}_h = \text{Spearman}\left(\hat{\alpha}_t,\, r_{t+h}\right)$$

We compute:
- **Rank IC (Spearman)** - robust to outliers, industry standard
- **Pearson IC** - sensitive to return distribution shape
- **p-value** - one-tailed test that IC > 0
- **IC t-stat** - $\text{IC} / (\sigma_{IC} / \sqrt{T})$


In [ ]:
def compute_ic_series(df: pd.DataFrame, horizon: int,
                      method: str = "spearman") -> pd.Series:
    """
    Compute daily cross-sectional IC for a given horizon.

    Parameters
    ----------
    df      : Panel with columns ['date','asset','signal', f'fwd_{h}d']
    horizon : Forward-return horizon in days
    method  : 'spearman' (rank IC) or 'pearson'
    """
    col = f"fwd_{horizon}d"

    def _ic_one_day(g):
        sig, ret = g["signal"].values, g[col].values
        mask = np.isfinite(sig) & np.isfinite(ret)
        if mask.sum() < 10:
            return np.nan
        if method == "spearman":
            ic, _ = spearmanr(sig[mask], ret[mask])
        else:
            ic, _ = pearsonr(sig[mask], ret[mask])
        return ic

    return df.groupby("date").apply(_ic_one_day).rename(f"IC_{horizon}d")


def compute_ic_table(df: pd.DataFrame, horizons: List[int]) -> pd.DataFrame:
    """Compute IC statistics across all horizons."""
    rows = []
    for h in horizons:
        ic_ts = compute_ic_series(df, h)
        T     = ic_ts.dropna().__len__()
        mu    = ic_ts.mean()
        sd    = ic_ts.std()
        se    = sd / np.sqrt(T)
        t_stat= mu / se if se > 0 else np.nan
        pval  = t_dist.sf(t_stat, df=T - 1)       # one-tailed p(IC > 0)
        icir  = mu / sd if sd > 0 else np.nan
        pct_pos = (ic_ts > 0).mean()
        rows.append({
            "Horizon (d)": h,
            "Mean IC":     mu,
            "Std IC":      sd,
            "ICIR":        icir,
            "t-stat":      t_stat,
            "p-value":     pval,
            "% IC > 0":    pct_pos,
            "Obs":         T,
        })
    return pd.DataFrame(rows).set_index("Horizon (d)")


# - Compute for all signals─
ic_tables: Dict[str, pd.DataFrame] = {}
for name, df in processed.items():
    ic_tables[name] = compute_ic_table(df, CFG.horizons)

# - Display momentum IC table─
print("- momentum_1m IC Summary")
display(ic_tables["momentum_1m"].style
    .background_gradient(subset=["Mean IC", "ICIR"], cmap="RdYlGn")
    .format({
        "Mean IC": "{:.4f}", "Std IC": "{:.4f}", "ICIR": "{:.3f}",
        "t-stat": "{:.2f}", "p-value": "{:.4f}", "% IC > 0": "{:.1%}",
    })
)


## 5. IC Information Ratio (ICIR)

$$\text{ICIR} = \frac{\overline{\text{IC}}}{\sigma_{\text{IC}}}$$

**Industry thresholds:**
| ICIR | Interpretation |
|------|---------------|
| > 0.5 | Good signal |
| > 1.0 | Strong signal |
| > 2.0 | Exceptional (verify for overfitting) |

ICIR is analogous to the Sharpe ratio but for signal quality.


In [ ]:
# - Cross-signal ICIR comparison at each horizon
icir_df = pd.DataFrame(
    {name: tbl["ICIR"] for name, tbl in ic_tables.items()}
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: ICIR heatmap
sns.heatmap(
    icir_df.T,
    annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, linewidths=0.5, ax=axes[0],
    cbar_kws={"label": "ICIR"},
)
axes[0].set_title("ICIR Heatmap - All Signals x Horizons", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Horizon (days)")
axes[0].set_ylabel("")

# Right: Mean IC comparison
ic_mean_df = pd.DataFrame(
    {name: tbl["Mean IC"] for name, tbl in ic_tables.items()}
)
for i, (col, color) in enumerate(zip(ic_mean_df.columns, COLORS.values())):
    axes[1].plot(ic_mean_df.index, ic_mean_df[col], "o-",
                 label=col, color=color, linewidth=2, markersize=7)

axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_xlabel("Forecast Horizon (days)")
axes[1].set_ylabel("Mean Spearman IC")
axes[1].set_title("Alpha Decay - Mean IC vs Horizon (All Signals)", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(mtick.FormatStrFormatter("%.3f"))

plt.tight_layout()
plt.show()


## 6. Alpha Decay Curve & Half-Life Estimation

We fit an **exponential decay model** to the IC-vs-horizon curve:

$$\text{IC}(h) = \text{IC}_0 \cdot e^{-\lambda h}$$

The **half-life** is:

$$t_{1/2} = \frac{\ln 2}{\lambda}$$

We also compute **95% confidence intervals** via bootstrapped parameter uncertainty.


In [ ]:
def exp_decay(h: np.ndarray, ic0: float, lam: float) -> np.ndarray:
    return ic0 * np.exp(-lam * np.asarray(h))


def fit_decay(ic_table: pd.DataFrame, signal_name: str) -> Dict:
    """Fit exponential decay and derive half-life with CI."""
    h_vals = np.array(ic_table.index, dtype=float)
    ic_vals = ic_table["Mean IC"].values

    try:
        popt, pcov = curve_fit(
            exp_decay, h_vals, ic_vals,
            p0=[ic_vals[0], 0.05],
            bounds=([0, 1e-6], [1.0, 10.0]),
            maxfev=5000,
        )
        ic0, lam = popt
        half_life = np.log(2) / lam

        # Delta-method CI for half-life
        perr  = np.sqrt(np.diag(pcov))
        lam_lo = max(lam - 1.96 * perr[1], 1e-6)
        lam_hi = lam + 1.96 * perr[1]
        hl_lo, hl_hi = np.log(2) / lam_hi, np.log(2) / lam_lo
    except RuntimeError:
        ic0, lam, half_life = np.nan, np.nan, np.nan
        hl_lo, hl_hi = np.nan, np.nan

    return dict(
        signal=signal_name, ic0=ic0, lam=lam,
        half_life=half_life, hl_lo=hl_lo, hl_hi=hl_hi,
    )


decay_fits = {name: fit_decay(ic_tables[name], name) for name in SIGNAL_SPECS}
decay_summary = pd.DataFrame(decay_fits).T.set_index("signal")
decay_summary = decay_summary.astype(float)

print("- Decay Fit Summary")
display(decay_summary.style
    .background_gradient(subset=["half_life"], cmap="RdYlGn_r")
    .format("{:.3f}")
)


In [ ]:
# - Decay curves with fitted lines
h_fine = np.linspace(1, max(CFG.horizons), 200)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes_flat = axes.ravel()

for ax, (name, spec) in zip(axes_flat, SIGNAL_SPECS.items()):
    color = list(COLORS.values())[list(SIGNAL_SPECS).index(name)]
    tbl   = ic_tables[name]
    fit   = decay_fits[name]

    ax.errorbar(
        tbl.index, tbl["Mean IC"],
        yerr=1.96 * tbl["Std IC"] / np.sqrt(tbl["Obs"]),
        fmt="o", color=color, markersize=8, capsize=4, label="Mean IC ± 95% CI",
    )

    if not np.isnan(fit["lam"]):
        y_fit = exp_decay(h_fine, fit["ic0"], fit["lam"])
        ax.plot(h_fine, y_fit, "--", color=color, linewidth=2,
                label=f"Fit: IC₀={fit['ic0']:.3f}, λ={fit['lam']:.3f}")
        ax.fill_between(h_fine,
            exp_decay(h_fine, fit["ic0"], np.log(2) / fit["hl_lo"]),
            exp_decay(h_fine, fit["ic0"], np.log(2) / fit["hl_hi"]),
            alpha=0.15, color=color)

    ax.axhline(0, color="gray", linestyle=":", linewidth=0.8)
    ax.set_title(f"{name}\nHalf-life = {fit['half_life']:.1f}d "
                 f"[{fit['hl_lo']:.1f}-{fit['hl_hi']:.1f}d]",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Horizon (days)", fontsize=9)
    ax.set_ylabel("Mean IC", fontsize=9)
    ax.legend(fontsize=8)

axes_flat[-1].set_visible(False)
plt.suptitle("Alpha Decay Curves - Exponential Fit with 95% Confidence Band",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Signal Autocorrelation & Persistence

Signal autocorrelation ρ(lag) directly governs **signal persistence**. A high ρ means the
signal is stable enough to trade at lower frequency. The link to half-life is:

$$t_{1/2} \approx -\frac{\ln 2}{\ln \rho}$$

We compute **cross-sectional mean absolute autocorrelation** for each lag.


In [ ]:
def signal_autocorrelation(df: pd.DataFrame, max_lag: int = 30) -> pd.Series:
    """
    Compute the cross-sectional mean absolute autocorrelation of the signal.
    For each lag k, average |corr(signal_t, signal_{t-k})| across assets.
    """
    pivot = df.pivot(index="date", columns="asset", values="signal")
    lags, mean_ac = [], []
    for lag in range(1, max_lag + 1):
        shifted = pivot.shift(lag)
        ac_per_asset = pivot.corrwith(shifted, method="spearman")
        lags.append(lag)
        mean_ac.append(ac_per_asset.abs().mean())
    return pd.Series(mean_ac, index=lags, name="MeanAbsAC")


fig, ax = plt.subplots(figsize=(13, 5))
for name, color in zip(SIGNAL_SPECS, COLORS.values()):
    ac = signal_autocorrelation(processed[name], max_lag=40)
    hl = decay_fits[name]["half_life"]
    ax.plot(ac.index, ac.values, "-", color=color, linewidth=2,
            label=f"{name} (HL≈{hl:.1f}d)")

ax.axhline(0.1, color="gray", linestyle="--", linewidth=0.8, label="AC=0.10 threshold")
ax.set_xlabel("Lag (trading days)", fontsize=11)
ax.set_ylabel("Mean |Spearman AC|", fontsize=11)
ax.set_title("Signal Persistence - Cross-Sectional Autocorrelation by Lag",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f"))
plt.tight_layout()
plt.show()

# Implied half-lives from autocorrelation
print("\n- Implied Half-Life from Autocorrelation─")
for name in SIGNAL_SPECS:
    pivot = processed[name].pivot(index="date", columns="asset", values="signal")
    rho_1 = pivot.corrwith(pivot.shift(1), method="spearman").mean()
    hl_ac = -np.log(2) / np.log(rho_1) if rho_1 > 0 else np.nan
    hl_fit = decay_fits[name]["half_life"]
    print(f"  {name:15s}  rho(1)={rho_1:.3f}  HL_AC={hl_ac:.1f}d  HL_fit={hl_fit:.1f}d")


## 8. Turnover & Cost-Adjusted IC

Even a high raw IC can be unprofitable once transaction costs are deducted.

**Cost-adjusted IC:**
$$\text{IC}^{\text{adj}}_h = \text{IC}_h - \frac{c \cdot \text{TO}_h}{\sigma_{r_h}}$$

where $c$ is the round-trip cost (bps) and $\text{TO}_h$ is the signal turnover.

**Daily turnover** at horizon $h$:
$$\text{TO}_h = \mathbb{E}\left[\left|\hat{\alpha}_{t+h} - \hat{\alpha}_t\right|\right] / 2$$


In [ ]:
def signal_turnover(df: pd.DataFrame) -> pd.Series:
    """
    Mean one-way absolute change in rank-normalised signal, per day.
    Represents the portfolio rebalancing fraction.
    """
    pivot = df.pivot(index="date", columns="asset", values="signal")
    daily_to = pivot.diff().abs().mean(axis=1)  # mean over assets
    return daily_to


def cost_adjusted_ic(
    mean_ic: float,
    turnover: float,
    ret_std: float,
    bps_cost: float = 5.0,
) -> float:
    """
    Estimate cost-adjusted IC.

    Parameters
    ----------
    mean_ic  : raw mean IC
    turnover : fraction of portfolio rebalanced per day (0-1)
    ret_std  : cross-sectional std dev of forward returns
    bps_cost : one-way transaction cost in bps
    """
    cost_drag = (bps_cost / 10_000) * turnover / max(ret_std, 1e-8)
    return mean_ic - cost_drag


rows = []
for name in SIGNAL_SPECS:
    df   = processed[name]
    to   = signal_turnover(df).mean()
    for h in CFG.horizons:
        raw_ic  = ic_tables[name].loc[h, "Mean IC"]
        ret_std = df[f"fwd_{h}d"].std()
        adj_ic  = cost_adjusted_ic(raw_ic, to, ret_std, CFG.bps_cost)
        rows.append(dict(signal=name, horizon=h, turnover=to,
                         raw_ic=raw_ic, adj_ic=adj_ic,
                         ic_erosion_pct=(raw_ic - adj_ic) / abs(raw_ic) if raw_ic != 0 else 0))

tca_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left - Turnover by signal
to_by_signal = tca_df.groupby("signal")["turnover"].first()
colors_list  = list(COLORS.values())[:len(to_by_signal)]
to_by_signal.plot.bar(ax=axes[0], color=colors_list, edgecolor="white")
axes[0].set_title("Daily Signal Turnover by Signal Type", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Mean Daily Turnover (fraction)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Right - Raw vs Adj IC heatmap (1d horizon)
pivot_raw = tca_df[tca_df["horizon"] == 5].set_index("signal")[["raw_ic", "adj_ic"]]
pivot_raw.columns = ["Raw IC (5d)", "Cost-Adj IC (5d)"]
pivot_raw.plot.bar(ax=axes[1], color=[COLORS["blue"], COLORS["orange"]],
                   edgecolor="white", width=0.6)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("Raw vs Cost-Adjusted IC at 5-Day Horizon", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Mean IC")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

print("- Cost-Adjusted IC Table (5d horizon, 5 bps one-way)─")
display(tca_df[tca_df["horizon"] == 5][["signal","turnover","raw_ic","adj_ic","ic_erosion_pct"]]
        .set_index("signal")
        .style.format({"turnover": "{:.3f}", "raw_ic": "{:.4f}",
                        "adj_ic": "{:.4f}", "ic_erosion_pct": "{:.1%}"})
        .background_gradient(subset=["adj_ic"], cmap="RdYlGn")
)


## 9. Rolling IC & Regime Overlay

Rolling IC reveals **signal regime dependence** - periods where the signal fires strongly vs.
periods of alpha attrition. We overlay a simple volatility-regime label (high / low vol).


In [ ]:
def rolling_ic_series(df: pd.DataFrame, horizon: int, window: int = 63) -> pd.Series:
    """Compute rolling cross-sectional Spearman IC over a rolling date window."""
    daily = compute_ic_series(df, horizon, method="spearman")
    return daily.rolling(window=window, min_periods=window // 2).mean()


# - Proxy market returns (mean cross-sectional return)
def market_vol_regime(df: pd.DataFrame, window: int = 21,
                      pct_threshold: float = 0.60) -> pd.Series:
    """Label each date as high-vol (1) or low-vol (0) regime."""
    daily_ret = df.groupby("date")["fwd_1d"].mean()
    vol = daily_ret.rolling(window).std()
    threshold = vol.quantile(pct_threshold)
    return (vol > threshold).astype(int).rename("vol_regime")


fig, axes = plt.subplots(len(SIGNAL_SPECS), 1, figsize=(16, 3.5 * len(SIGNAL_SPECS)),
                          sharex=True)

for ax, (name, color) in zip(axes, zip(SIGNAL_SPECS, COLORS.values())):
    df  = processed[name]
    ric = rolling_ic_series(df, horizon=5, window=CFG.rolling_window)
    reg = market_vol_regime(df)

    # Shade high-vol regime
    in_regime = False
    for i, (date, rv) in enumerate(reg.items()):
        if rv == 1 and not in_regime:
            start = date; in_regime = True
        elif rv == 0 and in_regime:
            ax.axvspan(start, date, alpha=0.12, color="red", label="High-Vol Regime" if i < 5 else "")
            in_regime = False

    ax.plot(ric.index, ric.values, color=color, linewidth=1.4, label=f"{name}")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.fill_between(ric.index, 0, ric.values, where=ric.values > 0,
                     alpha=0.2, color=color)
    ax.fill_between(ric.index, 0, ric.values, where=ric.values < 0,
                     alpha=0.2, color="red")
    ax.set_ylabel("Rolling IC", fontsize=9)
    ax.set_title(f"{name} - {CFG.rolling_window}d Rolling IC at 5-Day Horizon",
                 fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, loc="upper right")
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.3f"))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xlabel("Date", fontsize=11)
plt.suptitle("Rolling IC Time Series - All Signals (Red bands = High Volatility Regime)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 10. Optimal Holding Period

The **optimal holding period** maximises cost-adjusted ICIR:

$$h^* = \arg\max_h \frac{\text{IC}^{\text{adj}}_h}{\sigma_{\text{IC}_h}}$$

This balances signal strength against trading costs.


In [ ]:
optimal_rows = []
for name in SIGNAL_SPECS:
    df = processed[name]
    to = signal_turnover(df).mean()
    best_icir, best_h = -np.inf, None

    for h in CFG.horizons:
        tbl     = ic_tables[name]
        raw_ic  = tbl.loc[h, "Mean IC"]
        std_ic  = tbl.loc[h, "Std IC"]
        ret_std = df[f"fwd_{h}d"].std()
        adj_ic  = cost_adjusted_ic(raw_ic, to, ret_std, CFG.bps_cost)
        adj_icir = adj_ic / std_ic if std_ic > 0 else np.nan

        if np.isfinite(adj_icir) and adj_icir > best_icir:
            best_icir = adj_icir
            best_h    = h

    optimal_rows.append(dict(
        signal=name,
        optimal_horizon=best_h,
        adj_icir=best_icir,
        half_life=decay_fits[name]["half_life"],
    ))

opt_df = pd.DataFrame(optimal_rows).set_index("signal")

# - Bar chart of optimal horizons─
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

opt_df["optimal_horizon"].plot.bar(
    ax=axes[0], color=list(COLORS.values())[:len(opt_df)], edgecolor="white"
)
axes[0].set_title("Optimal Holding Period by Signal", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Optimal Horizon (days)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

# Scatter: half-life vs optimal horizon
for i, (idx, row) in enumerate(opt_df.iterrows()):
    color = list(COLORS.values())[i]
    axes[1].scatter(row["half_life"], row["optimal_horizon"],
                    s=180, color=color, label=idx, zorder=5)

lim_max = max(opt_df[["half_life","optimal_horizon"]].max()) * 1.1
axes[1].plot([0, lim_max], [0, lim_max], "k--", linewidth=0.8, label="y = x")
axes[1].set_xlabel("IC Half-Life (days)", fontsize=11)
axes[1].set_ylabel("Optimal Holding Period (days)", fontsize=11)
axes[1].set_title("Half-Life vs Optimal Holding Period", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("- Optimal Holding Period Summary─")
display(opt_df.style
    .background_gradient(subset=["adj_icir"], cmap="RdYlGn")
    .format({"optimal_horizon": "{:.0f}d", "adj_icir": "{:.3f}", "half_life": "{:.1f}d"})
)


## 11. Multi-Signal Ensemble Decay

A naïve equal-weight ensemble of orthogonal signals can improve ICIR through
diversification. We compute the ensemble IC and compare against individual signals.

$$\hat{\alpha}^{\text{ens}} = \frac{1}{K} \sum_{k=1}^{K} \hat{\alpha}_k$$


In [ ]:
# - Build ensemble signal─
ensemble_rows = []
for date in dates:
    sigs = {}
    for name, df in processed.items():
        sub = df[df["date"] == date][["asset", "signal"]].set_index("asset")
        sigs[name] = sub["signal"]

    sig_df = pd.DataFrame(sigs)
    sig_df.index.name = "asset"

    # Correlations between signals (cross-sectional)
    ens_signal = sig_df.mean(axis=1)

    base_row = {"date": date}
    # Retrieve forward returns from one panel (all share same returns structure)
    base_df = processed["momentum_1m"]
    rets = base_df[base_df["date"] == date].set_index("asset")
    for h in CFG.horizons:
        col = f"fwd_{h}d"
        if col in rets.columns:
            base_row[col] = rets[col]
            base_row["ens_signal"] = ens_signal
    ensemble_rows.append(base_row)

# Simpler approach: join all signals and compute ensemble IC per horizon
ens_by_horizon = {}
for h in CFG.horizons:
    col = f"fwd_{h}d"
    # Use momentum panel as return base
    base = processed["momentum_1m"][["date", "asset", col]].copy()
    for name, df in processed.items():
        base = base.merge(df[["date", "asset", "signal"]].rename(
            columns={"signal": f"sig_{name}"}),
            on=["date", "asset"], how="left")
    sig_cols = [c for c in base.columns if c.startswith("sig_")]
    base["ens_signal"] = base[sig_cols].mean(axis=1)

    def _ic(g):
        mask = g["ens_signal"].notna() & g[col].notna()
        if mask.sum() < 10: return np.nan
        return spearmanr(g.loc[mask, "ens_signal"], g.loc[mask, col])[0]

    ens_by_horizon[h] = base.groupby("date").apply(_ic).mean()

ens_ic = pd.Series(ens_by_horizon, name="Ensemble")

# - Compare ensemble vs individual
fig, ax = plt.subplots(figsize=(13, 5))

for name, color in zip(SIGNAL_SPECS, COLORS.values()):
    ax.plot(CFG.horizons, ic_tables[name]["Mean IC"].values,
            "o--", color=color, linewidth=1.5, alpha=0.7, label=name)

ax.plot(CFG.horizons, ens_ic.values, "s-",
        color="black", linewidth=3, markersize=10, label="Equal-Weight Ensemble")

ax.axhline(0, color="gray", linestyle=":", linewidth=0.8)
ax.set_xlabel("Forecast Horizon (days)", fontsize=11)
ax.set_ylabel("Mean Spearman IC", fontsize=11)
ax.set_title("Ensemble vs Individual Signal IC - Alpha Decay Comparison",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ICIR gain
ens_icir_rows = []
for h in CFG.horizons:
    best_ind = max(ic_tables[n].loc[h, "ICIR"] for n in SIGNAL_SPECS)
    ens_icir = ens_ic[h] / np.mean([ic_tables[n].loc[h, "Std IC"] for n in SIGNAL_SPECS])
    ens_icir_rows.append(dict(Horizon=h, Best_Individual_ICIR=best_ind, Ensemble_ICIR=ens_icir,
                               ICIR_Uplift=ens_icir - best_ind))

ens_icir_df = pd.DataFrame(ens_icir_rows).set_index("Horizon")
print("- Ensemble ICIR vs Best Individual─")
display(ens_icir_df.style
    .background_gradient(subset=["ICIR_Uplift"], cmap="RdYlGn")
    .format("{:.4f}")
)


## 12. Summary Dashboard

Consolidated view of all decay metrics in a single reference table.


In [ ]:
# - Build comprehensive summary─
summary_rows = []
for name in SIGNAL_SPECS:
    tbl = ic_tables[name]
    fit = decay_fits[name]
    opt = opt_df.loc[name]
    df  = processed[name]
    to  = signal_turnover(df).mean()

    # IC at key horizons
    ic_1d  = tbl.loc[1,  "Mean IC"] if 1  in tbl.index else np.nan
    ic_5d  = tbl.loc[5,  "Mean IC"] if 5  in tbl.index else np.nan
    ic_21d = tbl.loc[21, "Mean IC"] if 21 in tbl.index else np.nan

    icir_1d = tbl.loc[1,  "ICIR"] if 1  in tbl.index else np.nan
    icir_5d = tbl.loc[5,  "ICIR"] if 5  in tbl.index else np.nan

    summary_rows.append({
        "Signal":             name,
        "IC (1d)":            ic_1d,
        "IC (5d)":            ic_5d,
        "IC (21d)":           ic_21d,
        "ICIR (1d)":          icir_1d,
        "ICIR (5d)":          icir_5d,
        "IC₀":                fit["ic0"],
        "λ":                  fit["lam"],
        "Half-Life (d)":      fit["half_life"],
        "HL 95% CI":          f"[{fit['hl_lo']:.1f}, {fit['hl_hi']:.1f}]",
        "Daily Turnover":     to,
        "Optimal Horizon (d)":opt["optimal_horizon"],
    })

summary_df = pd.DataFrame(summary_rows).set_index("Signal")

display(summary_df.style
    .background_gradient(subset=["IC (1d)","IC (5d)","ICIR (1d)","ICIR (5d)"], cmap="RdYlGn")
    .background_gradient(subset=["Half-Life (d)"], cmap="Blues")
    .format({
        "IC (1d)": "{:.4f}", "IC (5d)": "{:.4f}", "IC (21d)": "{:.4f}",
        "ICIR (1d)": "{:.3f}", "ICIR (5d)": "{:.3f}",
        "IC₀": "{:.4f}", "λ": "{:.4f}", "Half-Life (d)": "{:.1f}",
        "Daily Turnover": "{:.3f}", "Optimal Horizon (d)": "{:.0f}",
    })
)


In [ ]:
# - Final interpretation banner─
print("=" * 70)
print("  ALPHA DECAY ANALYSIS - KEY TAKEAWAYS")
print("=" * 70)
for name in SIGNAL_SPECS:
    fit = decay_fits[name]
    opt = opt_df.loc[name]
    ic5 = ic_tables[name].loc[5, "ICIR"]
    freq = ("daily" if opt["optimal_horizon"] <= 3 else
            "weekly" if opt["optimal_horizon"] <= 7 else
            "bi-weekly" if opt["optimal_horizon"] <= 14 else "monthly")
    print(f"  {name:15s}  HL={fit['half_life']:5.1f}d  "
          f"ICIR(5d)={ic5:.3f}  "
          f"Optimal horizon={opt['optimal_horizon']:.0f}d -> {freq} rebalance")
print("=" * 70)
print()
print("Signals with half-life < 5d require daily rebalancing and low transaction costs.")
print("Signals with half-life > 20d can be traded weekly or monthly.")
print("The ensemble consistently improves ICIR through orthogonal diversification.")
